<a href="https://colab.research.google.com/github/manlikegoose/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

In [36]:

def clean_machakos(machakos_raw):
    clean = []
    skipped = []

    for month, value in enumerate(machakos_raw):
        try:
            clean.append(int(value.strip()))
        except ValueError:
            skipped.append(month)


    average = round(np.mean(clean))

    for month in skipped:
        clean.insert(month, average)


    for month in skipped:
        print(f"{months[month]} was assigned with {average} mm")

    return clean


machakos_clean = clean_machakos(machakos_raw)

print("Clean Machakos data:", machakos_clean)


April was assigned with 68 mm
September was assigned with 68 mm
Clean Machakos data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [37]:
clean_rows.append(machakos_clean)

R = np.array(clean_rows)

print("Rainfall array:")
print(R)

print("Shape:", R.shape)
print("Data type:", R.dtype)



Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Data type: int64


In [38]:

annual_totals = np.sum(R, axis=1)
monthly_averages = np.mean(R, axis=0)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nMonthly average rainfall:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.2f} mm")


Annual rainfall totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Monthly average rainfall:
January: 47.00 mm
February: 47.60 mm
March: 91.00 mm
April: 164.40 mm
May: 171.20 mm
June: 67.60 mm
July: 56.60 mm
August: 58.20 mm
September: 64.20 mm
October: 67.80 mm
November: 127.40 mm
December: 80.60 mm


In [39]:

wettest_indices = np.argmax(R, axis=1)
wettest_months = np.array(months)[wettest_indices]

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")


Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [40]:

dry_mask = R < 50

dry_month_counts = np.sum(dry_mask, axis=1)

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")


Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


In [41]:

consecutive_rainfall = R[:, :-1] + R[:, 1:]

planting_indices = np.argmax(consecutive_rainfall, axis=1)

planting_windows = [
    f"{months[i]}–{months[i+1]}"
    for i in planting_indices
]

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")


Nairobi: April–May
Nakuru: April–May
Kisumu: April–May
Mombasa: April–May
Machakos: November–December


In [42]:

import csv

with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    for row in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow(row)

print("rainfall_summary.csv created successfully!")


rainfall_summary.csv created successfully!



## Planting Advisory

The rainfall analysis identifies different rainfall patterns across the five counties, meaning planting schedules should be tailored to each location. Farmers in each county should consider the recommended two-month planting window, which represents the period with the highest combined rainfall. Nairobi and Nakuru farmers should also consider the distribution of rainfall throughout the year when planning their planting activities. Farmers should monitor local weather forecasts before planting because historical rainfall patterns do not guarantee future rainfall. The Machakos dataset contained two invalid entries, which were replaced using the rounded average of valid rainfall readings. Therefore, its results should be interpreted with caution because imputation may affect the accuracy of the analysis.
